In [1]:
import fdtdx
import jax
import jax.numpy as jnp
from matplotlib import pyplot as plt
import pytreeclass as tc
from IPython.display import Video
%matplotlib inline
import numpy as np

In [ ]:

def build_and_run_single_lambda(lam_m, output_wg_gap):
    
    N_CORE  = jnp.sqrt(12.0)
    N_BG    = 1.0
    DX      = 20e-9

    GRID_X_LENGTH = 6.5e-6
    GRID_Y_LENGTH = 12e-6
    GRID_Z_LENGTH = 1.5e-6  

    BUS_LENGTH = 2.0e-6
    BUS_WIDTH  = 0.2e-6

    NUM_OUTPUT_WG = 2
    OUTPUT_WG_LENGTH_X      = 2.4e-6
    OUTPUT_WG_WIDTH         = 0.2e-6
    OUTPUT_WG_GAP           = output_wg_gap
    OUTPUT_WG_CENTER_SPACING = OUTPUT_WG_WIDTH + OUTPUT_WG_GAP

    DUMMY_LENGTH_Z = 0.5e-6   

    period = fdtdx.constants.wavelength_to_period(lam_m)

    # =============================================================================
    # Colors 
    # =============================================================================
    COLOR_WG        = fdtdx.colors.LIGHT_BLUE
    COLOR_TAPERRECT = fdtdx.colors.TAN
    COLOR_OPTREG    = fdtdx.colors.MAGENTA
    COLOR_SOURCE    = fdtdx.colors.ORANGE
    COLOR_DET       = fdtdx.colors.LIGHT_GREEN

    # =============================================================================
    # Materials
    # =============================================================================
    material_config = {
        "mat_bg":   fdtdx.Material(permittivity=fdtdx.constants.relative_permittivity_air),
        "mat_core": fdtdx.Material(permittivity=float(N_CORE**2)),
    }

    # =============================================================================
    # Simulation config + volume + PML
    # =============================================================================
    config = fdtdx.SimulationConfig(
        time=300e-15,
        resolution=float(DX),
        dtype=jnp.float32,
        courant_factor=0.99,
    )

    volume = fdtdx.SimulationVolume(
        partial_real_shape=(float(GRID_X_LENGTH), float(GRID_Y_LENGTH), float(GRID_Z_LENGTH)),
        material=material_config["mat_bg"],
    )

    object_list = [volume]
    placement_constraints = []

    bound_cfg = fdtdx.BoundaryConfig.from_uniform_bound(thickness=10, boundary_type="pml")
    boundaries, b_constraints = fdtdx.boundary_objects_from_config(bound_cfg, volume)
    object_list.extend(boundaries.values())
    placement_constraints.extend(b_constraints)

    # =============================================================================
    # Bus waveguide (left-aligned to volume)
    # =============================================================================
    bus = fdtdx.UniformMaterialObject(
        name="bus_waveguide",
        material=material_config["mat_core"],
        partial_real_shape=(float(BUS_LENGTH), float(BUS_WIDTH), float(DUMMY_LENGTH_Z)),
        color=COLOR_WG,
    )

    placement_constraints.append(
        bus.place_relative_to(
            boundaries['min_x'],
            axes=(0, 1, 2),
            own_positions=(-1, 0, 0),
            other_positions=(-1, 0, 0),
        )
    )
    object_list.append(bus)

    # =============================================================================
    # Wedge Area: taper (isosceles trapezoid) + rectangle region (same color)
    # =============================================================================
    L  = 0.5e-6     # taper length (x)
    w1 = 0.2e-6     # width at left face
    w2 = 10.0e-6    # width at right face
    Lz = float(DUMMY_LENGTH_Z)

    verts_centered = np.array(
        [
            (-L/2, -w1/2),
            (-L/2,  w1/2),
            ( L/2,  w2/2),
            ( L/2, -w2/2),
        ],
        dtype=np.float32,
    )

    taper_Lx = L 
    taper_Ly = max(w1, w2) 

    taper = fdtdx.ExtrudedPolygon(
        name="taper",
        materials=material_config,
        material_name="mat_core",
        axis=2,
        vertices=jnp.array(verts_centered, dtype=jnp.float32),
        partial_real_shape=(float(taper_Lx), float(taper_Ly), float(Lz)),
        color=COLOR_TAPERRECT,
    )
    object_list.append(taper)

    placement_constraints.append(
        taper.place_relative_to(
            bus,
            axes=(0, 1, 2),
            own_positions=(-1, 0, 0),
            other_positions=(+1, 0, 0),
        )
    )

    rectangle_length_x = 0.5e-6
    rectangle_width_y  = 10.0e-6

    rectangle = fdtdx.UniformMaterialObject(
        name="rectangle",
        material=material_config["mat_core"],
        partial_real_shape=(float(rectangle_length_x), float(rectangle_width_y), float(DUMMY_LENGTH_Z)),
        color=COLOR_TAPERRECT,
    )
    object_list.append(rectangle)

    placement_constraints.append(
        rectangle.place_relative_to(
            taper,
            axes=(0, 1, 2),
            own_positions=(-1, 0, 0),
            other_positions=(+1, 0, 0),
        )
    )

    # =============================================================================
    # Optimization region (Device)
    # =============================================================================

    optimization_region = fdtdx.UniformMaterialObject(
    name="optimization_region",
    material=material_config["mat_core"],
    partial_real_shape=(1.0e-6, 10.0e-6, float(DUMMY_LENGTH_Z)),
    color=COLOR_OPTREG,
    )
    object_list.append(optimization_region)

    placement_constraints.append(
        optimization_region.place_relative_to(
            rectangle,
            axes=(0, 1, 2),
            own_positions=(-1, 0, 0),
            other_positions=(+1, 0, 0),
        )
    )


    # =============================================================================
    # Output waveguides
    # =============================================================================
    output_wgs = []
    for i in range(NUM_OUTPUT_WG):
        out = fdtdx.UniformMaterialObject(
            name=f"out_wg_{i:02d}",
            material=material_config["mat_core"],
            partial_real_shape=(
                float(OUTPUT_WG_LENGTH_X),
                float(OUTPUT_WG_WIDTH),
                float(DUMMY_LENGTH_Z),
            ),
            color=COLOR_WG,
        )
        output_wgs.append(out)
        object_list.append(out)

    mid = (NUM_OUTPUT_WG - 1) / 2.0
    for i, out in enumerate(output_wgs):
        y_c = (i - mid) * OUTPUT_WG_CENTER_SPACING
        placement_constraints.append(
            out.place_relative_to(
                optimization_region,
                axes=(0, 1, 2),
                own_positions=(-1, 0, 0),
                other_positions=(+1, 0, 0),
                margins=(0.0, float(y_c), 0.0),
            )
        )

    # =============================================================================
    # Source
    # =============================================================================

    source = fdtdx.ModePlaneSource(
        name="input_source",
        wave_character=fdtdx.WaveCharacter(wavelength=float(lam_m)),
        #temporal_profile=temporal,
        direction="+",
        partial_grid_shape=(1, 20, 50),
        mode_index=0,
        color=COLOR_SOURCE,
    )
    object_list.append(source)

    placement_constraints.append(
        source.place_relative_to(
            volume,
            axes=(0, 1, 2),
            other_positions=(-1, 0, 0),
            own_positions=(-1, 0, 0),
            margins=(0.3e-6, 0.0, 0.0),
        )
    )

    # =============================================================================
    # Detectors
    # =============================================================================

    output_detectors = [] 
    for i, out in enumerate(output_wgs):
        det = fdtdx.ModeOverlapDetector(
        name=f"det_out_{i:02d}",
        wave_characters=[fdtdx.WaveCharacter(wavelength=lam_m)],
        direction="+",
        filter_pol=None,   # or "te"/"tm" if you want
        reduce_volume=False,   # keep it; overlap uses internal sum anyway
        dtype=jnp.complex64,
        #partial_real_shape=det_plane_shape_out,   # cross-section plane
        partial_grid_shape=(1, 20, 50),  # plane normal to propagation axis
        switch=fdtdx.OnOffSwitch(period=period, start_time=150e-15, on_for_periods=20)
    )

        object_list.append(det)
        output_detectors.append(det)

        placement_constraints += [
        # put the detector at the output face of the waveguide in +x direction
        det.place_relative_to(out, axes=(0,), other_positions=(+1,), own_positions=(+1,), margins=(-0.4e-6,)),
        # align center in y and z
        det.place_at_center(out, axes=(1, 2)),
        ]




    # =============================================================================
    # Place objects, then apply a binary mask to the optimization region
    # =============================================================================
    key = jax.random.PRNGKey(7)
    key_place, key_mask = jax.random.split(key, 2)
    objects, arrays, params, config, _ = fdtdx.place_objects(
        object_list=object_list,
        config=config,
        constraints=placement_constraints,
        key=key_place,
    )



    arrays, new_objects, _ = fdtdx.apply_params(
        arrays=arrays,
        objects=objects,
        params=params,
        key=key,
    )

    final_state = fdtdx.run_fdtd(arrays, new_objects, config, key_place)
    _, arrays_out = final_state

    # -----------------------------
    # 5) Compute T for this single lambda
    # -----------------------------
    ds = arrays_out.detector_states
    alpha_out = []
    for i in range(NUM_OUTPUT_WG):
        det = next(d for d in new_objects.forward_detectors if d.name == f"det_out_{i:02d}")
        alpha_out.append(det.compute_overlap(ds[det.name]))
    alpha_out = jnp.stack(alpha_out, axis=0)


    return {
    "lambda_m": float(lam_m),
    "T": alpha_out,
    "arrays_out": arrays_out,
    "objects": objects,
}



# -----------------------------
# Sweep output gap at fixed wavelength
# -----------------------------
results = []

lam_m_fixed = 1.55e-6
gap_um_list = np.linspace(0.1, 1.0, 10)
gap_m_list = gap_um_list * 1e-6

T_results = []

for gap_m in gap_m_list:

    out = build_and_run_single_lambda(
        lam_m=lam_m_fixed,
        output_wg_gap=gap_m,
    )

    T_cpu = np.asarray(jax.device_get(jnp.abs(out["T"])))
    T_results.append(T_cpu)

    del out

gap_um = gap_um_list
T_np = np.squeeze(np.stack(T_results))

# Make sure plotting array has shape (n_det, Ngap)
if T_np.ndim == 1:
    T_plot = T_np[None, :]
elif T_np.shape[0] == len(gap_um):
    T_plot = T_np.T
else:
    T_plot = T_np

n_det = T_plot.shape[0]

# -----------------------------
# Print results
# -----------------------------
for i in range(n_det):
    print(f"\nDetector det_out_{i:02d}")
    for g, t in zip(gap_um, T_plot[i, :]):
        print(f"gap = {g:.3f} µm   T = {t:.6f}")

# -----------------------------
# Plot results
# -----------------------------
plt.figure(figsize=(6,4))
for i in range(n_det):
    plt.plot(gap_um, T_plot[i, :], marker='o', label=f"det_out_{i:02d}")

plt.xlabel("Output waveguide gap (µm)")
plt.ylabel("|overlap|")
plt.title("Transmission vs output gap at λ = 1.55 µm")
plt.grid(True)
plt.legend()
plt.show()
